In [58]:
from dotenv import load_dotenv
load_dotenv() 

True

In [59]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

> 1. Load Data

In [60]:
loader = PyPDFLoader("../data/Artificial Intelligence Overview.pdf")
docs = loader.load()
length = len(docs)
length

2

> 2. Split Data

In [61]:
splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
split_docs = splitter.split_documents(docs)
split_length = len(split_docs)
split_length

2

> 3. Embedd data

In [62]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

> 4. Store Embddings in vector Database 

In [63]:
vector_store = Chroma.from_documents(
    documents=split_docs, 
    embedding=embeddings, 
    collection_name="ai_docs",
    persist_directory="../data/chroma_db_ai"
    )

In [64]:
query  = "Artificial Intelligence and generative AI content."

In [65]:
data = vector_store.similarity_search(query=query , k = 10)
len(data)
data

[Document(id='f3a5e8e9-d9c8-4c5c-9cbc-6bee3664ce93', metadata={'page': 1, 'producer': 'Skia/PDF m136', 'creationdate': '2026-10-01T08:34:45+00:00', 'title': 'Artificial Intelligence: Executive Overview', 'total_pages': 2, 'page_label': '2', 'creator': 'Chromium', 'moddate': '2026-10-01T08:34:45+00:00', 'source': '../data/Artificial Intelligence Overview.pdf'}, page_content='Artiﬁcial Intelligence Overview Page 1 of 1'),
 Document(id='40c56a7d-752c-4420-804d-6973399f321d', metadata={'page': 1, 'creator': 'Chromium', 'total_pages': 2, 'producer': 'Skia/PDF m136', 'page_label': '2', 'source': '../data/Artificial Intelligence Overview.pdf', 'title': 'Artificial Intelligence: Executive Overview', 'creationdate': '2026-10-01T08:34:45+00:00', 'moddate': '2026-10-01T08:34:45+00:00'}, page_content='Artiﬁcial Intelligence Overview Page 1 of 1'),
 Document(id='65ca7d02-940b-4a51-9d37-159b79e189fa', metadata={'page': 1, 'total_pages': 2, 'title': 'Artificial Intelligence: Executive Overview', 'cre

> Context to be provide to llm

In [66]:
context = ""
for doc in data:
    context += doc.page_content + "\n"

print(context)

Artiﬁcial Intelligence Overview Page 1 of 1
Artiﬁcial Intelligence Overview Page 1 of 1
Artiﬁcial Intelligence Overview Page 1 of 1
Artificial Intelligence
A concise brieﬁng on architectural paradigms, industry deployment metrics, and capabilities.
EXECUTIVE SUMMARY
Artiﬁcial Intelligence (AI) represents the simulation of human intelligence by computer systems. Modern
developments are driven by deep learning networks, large language models (LLMs), and scalable compute
infrastructure, enabling rapid transition from research frameworks to critical enterprise operations.
Market & Growth Snapshot
37.3%
CAGR (2023–2030)
Global Market Expansion
72%
ENTERPRISE ADOPTION
Integrating AI in ≥1 Function
10x
COMPUTE SCALING
Model Training Power Increase
Core Paradigms
Machine Learning Predictive
Statistical models that improve performance through
experience without explicit programming. Essential for
forecasting and structured data classiﬁcation.
Deep Learning Perceptual
Multi-layered neural networ

In [67]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [68]:
# res = llm.invoke(f""""can you provide a summary of the 
# following content: {context} and question: {query}""")

In [69]:
print(res)

content=[{'type': 'text', 'text': 'Based on the provided context:\n\n* **Artificial Intelligence (AI)** is broadly defined as the simulation of human intelligence by computer systems, driven by deep learning networks, large language models, and scalable compute infrastructure. \n* **Generative AI** is a specific core paradigm within AI focused on **synthesis**. It refers to models capable of creating text, images, code, and audio from complex underlying distributions, led by transformer architectures.', 'extras': {'signature': 'EmAKXgFpFH0TwcPXOPKY/ZRU4BDr8HWYZyDrMMXYVuUxE9HcjqLWg0PxDlSh4q9ho99vyQab+eeaKtIyI/8B6Q35F0OrdFRUlIpqerAy/RVwTTVUBcUT9pY+7cU+pXqEYzQ='}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a0f6d5-fdcc-7201-bebd-837eace9ad86-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 819, 'output_tokens': 88, 'total_tokens': 907, 'i

In [70]:
print(res.content[-1]['text'])

Based on the provided context:

* **Artificial Intelligence (AI)** is broadly defined as the simulation of human intelligence by computer systems, driven by deep learning networks, large language models, and scalable compute infrastructure. 
* **Generative AI** is a specific core paradigm within AI focused on **synthesis**. It refers to models capable of creating text, images, code, and audio from complex underlying distributions, led by transformer architectures.


#### Langchain - context-generate -> prompt -> llm -> string output

In [71]:
def get_context_from_query(query, vector_store, k=5):
    data = vector_store.similarity_search(query=query , k=k)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"
    return {
        "context": context,
        "question": query
    } 

In [72]:
prompt = PromptTemplate.from_template("""
    You are a helpful AI assistant.
    Use the following context to answer the question at the end. 
    If you don't know the answer,
    just say that you don't know, 
    don't try to make up an answer.
    Context: {context}
    Question: {question}
    """)

In [73]:
rag_chain = get_context_from_query | prompt | llm

In [74]:
res = rag_chain.invoke("what is the difference between AI and generative AI?", vector_store=vector_store)

In [75]:
print(res.content[-1]['text'])

Based on the provided context:

* **Artificial Intelligence (AI)** is broadly defined as the simulation of human intelligence by computer systems, driven by deep learning networks, large language models (LLMs), and scalable compute infrastructure. 
* **Generative AI** is a specific core paradigm within AI focused on synthesis—specifically, models capable of creating text, images, code, and audio from complex underlying distributions, led by transformer architectures.
